# M2 · Paso 2 — Entrenamiento en dos fases (`src/training.py`)
**TFM · HAM10000** · Responsable: **Alex Reinoso** · MLflow: `sandbox` (prueba) o `m2-entrenamiento` (real)

Un run = una arquitectura × una semilla: **fase 1** (base congelada) → **fase 2** (30 % final descongelado, BatchNormalization congelada), con `training.deterministic: true` (D10). Todo el código está en `src/training.py`; este notebook solo configura la sesión y lo lanza.

**Dos modos, un único interruptor (`MODO`):**
| Modo | Experimento | Qué hace |
|---|---|---|
| `"sandbox"` | `sandbox` | Prueba con pocas épocas (`SANDBOX_EPOCAS`) y, opcionalmente, pocos lotes. Mide el **tiempo real por paso** y proyecta el coste de los 9 runs (condición pendiente de D10). |
| `"real"` | `m2-entrenamiento` | Uno de los 9 runs oficiales. Exige git limpio y no admite acortar nada. |

**Antes de empezar:** T4 GPU, carpeta `TFM` como acceso directo en Drive, secretos `DAGSHUB_USER` y `DAGSHUB_TOKEN` activados en este notebook, y la rama `m2/entrenamiento` subida a GitHub. Guía: `docs/modulo2_guia_colab.md`.

## 0. Parámetros

In [ ]:
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
BRANCH = "m2/entrenamiento"           # en modo real, la rama de la que salen los resultados (normalmente main tras el PR)
AUTOR  = "alex"                       # alex | adrian | christian (tag `autor` del run)

MODO   = "sandbox"                    # "sandbox" | "real"
ARCHS  = ["resnet50", "densenet121", "efficientnetb0"]   # sandbox: las 3 para medir el coste; real: las de tu cuenta
SEED   = 42                           # real: 42 (Alex), 123 (Adrián) o 2024 (Christian) según docs/04_colab.md §7

# Solo sandbox ------------------------------------------------------------------------------
SANDBOX_EPOCAS    = (2, 2)            # (fase 1, fase 2)
SANDBOX_MAX_LOTES = None              # None = datos completos (mide el tiempo real); p. ej. 20 para una prueba más corta
RETOMAR_SANDBOX   = False             # False: borra el estado de pruebas anteriores y empieza de cero

# Solo real ---------------------------------------------------------------------------------
CONFIRMO_REAL = False                 # ponerlo en True a propósito; evita lanzar un run oficial por descuido

## 1. Configuración de la sesión

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU (entorno CPU)"
import sys; print("Python", sys.version.split()[0])
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -3
!pip install -q -r requirements.txt

In [ ]:
import shutil, zipfile, time
from pathlib import Path
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not LOCAL.exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time()-t0:.0f} s)")
assert len(list((LOCAL / "images").glob("*.jpg"))) == 10015
assert file_sha256(cfg["paths"]["metadata_csv"]) == "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"
assert file_sha256("data/splits/split_v1.csv") == "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"
print("[+] Datos y split v1 verificados | deterministic =", cfg["training"]["deterministic"])

In [ ]:
import mlflow
from google.colab import userdata
from src.tracking import setup_mlflow, git_dirty
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
print("MLflow:", setup_mlflow(cfg))
print("git_dirty =", git_dirty())
assert MODO in ("sandbox", "real")
if MODO == "real":
    assert CONFIRMO_REAL, "MODO='real': pon CONFIRMO_REAL = True si de verdad quieres lanzar un run oficial"
    assert not git_dirty(), "Hay cambios sin commit: este run no valdría como resultado final"
print("Modo:", MODO, "| experimento:", cfg["mlflow"]["experiments"]["m2_train" if MODO == "real" else "sandbox"])

## 2. Lanzar
- **Sandbox:** por cada arquitectura se entrenan `SANDBOX_EPOCAS` épocas por fase. Sin `SANDBOX_MAX_LOTES` se usa el train completo (el primer epoch incluye decodificar los 7 009 JPEG; los siguientes salen de la caché).
- **Reanudar:** si Colab se desconecta o paras la celda, **vuelve a ejecutar esta celda tal cual** (con `RETOMAR_SANDBOX = True` en sandbox): el run de MLflow se retoma (`Retomando run…`), las fases ya completadas no se repiten y la fase en curso sigue desde su última época.
- Cada run termina registrando curvas, matriz de confusión, `classification_report_val.json`, `predicciones_val.csv` y el modelo (`modelo/`).

In [ ]:
import shutil
from src.training import lanzar_run
resultados = {}
for arch in ARCHS:
    if MODO == "sandbox" and not RETOMAR_SANDBOX:
        shutil.rmtree(Path(cfg["paths"]["checkpoints_dir"]) / "sandbox" / f"m2-{arch}-s{SEED}-sandbox", ignore_errors=True)
    print(f"\n{'='*20} {arch} · semilla {SEED} · {MODO} {'='*20}")
    kw = dict(epochs=SANDBOX_EPOCAS, max_lotes=SANDBOX_MAX_LOTES) if MODO == "sandbox" else {}
    resultados[arch] = lanzar_run(arch, SEED, cfg, MODO, AUTOR, **kw)
    m = resultados[arch]["metricas"]
    print(f"[+] {resultados[arch]['run_name']}  run_id={resultados[arch]['run_id']}  "
          f"({resultados[arch]['tiempo_total_s']/60:.1f} min)  "
          f"val_macro_f1={m['macro_f1']:.3f}  val_mel_sensitivity={m['mel_sensitivity']:.3f}")

## 3. Proyección de coste (condición pendiente de D10) — solo sandbox
Se usa el tiempo por paso de la **última época de cada fase** (con la caché ya caliente) y se proyecta el peor caso: 15 + 30 épocas × 220 pasos, sin parada anticipada. **Condición:** si la suma por cuenta (las 3 redes con una semilla) supera ≈ 120 min, se vuelve a plantear `deterministic` al grupo.
No incluye la pasada por validación de cada época (1 502 imágenes solo hacia delante: del orden del 5–10 % de una época de entrenamiento, de 7 009 imágenes con retropropagación; súmale ese margen al leer el resultado).

In [ ]:
import pandas as pd
from src.training import proyeccion_tiempo
if MODO == "sandbox":
    s = {}
    for arch, r in resultados.items():
        h = pd.DataFrame(r["historial"])
        s[arch] = {"f1": h[h.fase == "f1"].iloc[-1].s_por_paso, "f2": h[h.fase == "f2"].iloc[-1].s_por_paso}
    tabla = proyeccion_tiempo(s)
    display(tabla)
    total = tabla.loc["TOTAL por cuenta (3 redes)", "min_por_run_peor_caso"]
    print(f"Peor caso por cuenta: {total:.0f} min -> ", "dentro del límite de ≈120 min: se mantiene deterministic=True" if total <= 120
          else "SUPERA el límite: llevarlo al grupo (D10)")

## 4. Comprobar que M3 podrá cargar el modelo
Se carga el modelo registrado por la ruta del run (en el flujo real, el Módulo 3 lo cargará por alias `ham10000-<arch>@final`; los alias se asignan en el paso de selección) y se comprueba que da las mismas probabilidades que el entrenamiento.

In [ ]:
import numpy as np
from src.data import make_dataset
for arch, r in resultados.items():
    modelo = mlflow.tensorflow.load_model(f"runs:/{r['run_id']}/modelo")
    x, _ = next(iter(make_dataset("val", arch, training=False, cache=False)))
    p = modelo.predict(x, verbose=0)
    ok = np.allclose(p, r["probs"][: len(p)], atol=1e-4)
    print(arch, "-> modelo cargado", modelo.output_shape, "| mismas probabilidades que en el entrenamiento:", ok)

## 5. Cierre
- Revisa en DagsHub → Experiments el run (curvas `f1_*` y `f2_*` en una sola gráfica, tags, parámetros `deterministic`, `gpu`, `tf_version`).
- En **real**, al terminar los 9 runs hay que marcar a mano `estado=final` (o `descartado`) en cada uno; no se hace automáticamente.
- *Entorno de ejecución → Desconectar y eliminar el entorno de ejecución.*